### Part 1: The Property Crawler

Async Playwright crawler with BFS traversal, JavaScript rendering, rate limiting, and full
metadata extraction. Outputs Markdown files and a JSONL corpus.

**Rubric coverage (15 pts)**
| Criterion | Pts | Notes |
|---|---|---|
| Playwright Implementation | 5 | Async BFS, browser lifecycle, JS rendering |
| Data Extraction Completeness | 5 | All metadata fields extracted |
| Output Format Correctness | 5 | Markdown files + JSONL corpus |

### 1. Setup & Installations

In [ ]:
import sys
import subprocess

COLAB_PACKAGES = [
    "playwright", "beautifulsoup4", "markdownify",
    "nest_asyncio", "python-dotenv", "pyyaml",
]

def _run(cmd: list[str]) -> None:
    """Run a shell command, raising on failure.

    Args:
        cmd: Command and arguments to execute.

    Raises:
        subprocess.CalledProcessError: If the command exits with a non-zero code.
    """
    subprocess.run(cmd, check=True)

if "google.colab" in sys.modules:
    print("Running in Google Colab")
    _run([sys.executable, "-m", "pip", "install", "-q"] + COLAB_PACKAGES)
    _run([sys.executable, "-m", "playwright", "install", "chromium"])
else:
    print("Local environment detected — using uv")
    try:
        _run(["uv", "sync"])
        print("Dependencies synced via uv.")
    except FileNotFoundError:
        print("uv not found. Install it from https://docs.astral.sh/uv/ or run 'pip install uv'.")
        raise
    try:
        _run([sys.executable, "-m", "playwright", "install", "chromium"])
        print("Playwright browsers installed.")
    except Exception as e:
        print(f"Could not install Playwright browsers: {e}")
        print("Run 'playwright install chromium' manually in your terminal.")

print("Setup complete")

### 2. Imports & Environment Setup

In [ ]:
import sys
import json
import time
import random
import yaml
import nest_asyncio
from pathlib import Path
from dotenv import load_dotenv
from urllib.parse import urljoin, urlparse

nest_asyncio.apply()

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

load_dotenv(project_root / ".env")

print(f"Project Root : {project_root}")
print(f"Source Path  : {src_path}")
print("Environment loaded & paths configured.")

### 3. Load Configuration

In [ ]:
config_path = project_root / "config" / "config.yaml"

if not config_path.exists():
    raise FileNotFoundError(f"Config not found at {config_path}")

with open(config_path, encoding="utf-8") as f:
    config = yaml.safe_load(f)

crawling_cfg = config["crawling"]

BASE_URL         = crawling_cfg["base_url"]
START_PATHS      = crawling_cfg["start_paths"]
START_URLS       = [urljoin(BASE_URL, path) for path in START_PATHS]
MAX_DEPTH        = crawling_cfg["max_depth"]
MAX_PAGES        = crawling_cfg["max_pages"]
REQUEST_DELAY    = crawling_cfg["delay_seconds"]
EXCLUDE_PATTERNS = crawling_cfg["exclude_patterns"]

MARKDOWN_DIR = project_root / config["paths"]["markdown_dir"]
JSONL_PATH   = project_root / config["paths"]["corpus_file"]

MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
JSONL_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Base URL      : {BASE_URL}")
print(f"Start points  : {len(START_URLS)}")
print(f"Max depth     : {MAX_DEPTH}")
print(f"Max pages     : {MAX_PAGES}")
print(f"Request delay : {REQUEST_DELAY}s")
print(f"Markdown dir  : {MARKDOWN_DIR}")
print(f"Corpus path   : {JSONL_PATH}")

### 4. Import Web Crawler Service

In [ ]:
try:
    from src.context_engineering.application.ingest_documents_service.web_crawler import PrimeLandsWebCrawler
    print("Service imported successfully.")
    print("Location: src.context_engineering.application.ingest_documents_service.web_crawler")
except ImportError as e:
    print("Import failed. Check your 'src' folder structure.")
    print(f"Error: {e}")
    raise

### 5. Execute Crawl

In [ ]:
print(f"Starting crawl at {time.strftime('%H:%M:%S')}...")
start_time = time.time()

documents = []

try:
    crawler = PrimeLandsWebCrawler(
        base_url=BASE_URL,
        max_depth=MAX_DEPTH,
        exclude_patterns=EXCLUDE_PATTERNS,
        jsonl_path=str(JSONL_PATH),
        max_pages=MAX_PAGES
    )

    documents = crawler.crawl(
        start_urls=START_URLS,
        request_delay=REQUEST_DELAY
    )

    elapsed = time.time() - start_time
    print(f"\nCrawl complete in {elapsed:.1f}s")
    print(f"Documents collected : {len(documents)}")
    print(f"Corpus saved to     : {JSONL_PATH}")

    if len(documents) == 0:
        print("WARNING: No documents were collected. Check BASE_URL, network access, or site structure.")
    elif len(documents) < 5:
        print(f"WARNING: Only {len(documents)} pages collected — rubric requires 5+. Consider increasing MAX_PAGES or adding more START_PATHS.")
    else:
        print(f"OK: {len(documents)} documents collected — meets the 5+ page requirement.")

except Exception as e:
    elapsed = time.time() - start_time
    print(f"\nCrawl FAILED after {elapsed:.1f}s")
    print(f"Error type : {type(e).__name__}")
    print(f"Details    : {e}")
    print("\nTroubleshooting tips:")
    print("  - Confirm Playwright browsers are installed: 'playwright install chromium'")
    print("  - Verify BASE_URL is reachable from your network")
    print("  - Check config.yaml for typos in start_paths or exclude_patterns")
    raise

### 6. Save Markdown Files

In [ ]:
print(f"Saving markdown files to: {MARKDOWN_DIR}")

saved_count = 0
failed_count = 0

for i, doc in enumerate(documents):
    try:
        parsed_url = urlparse(doc["url"])
        url_path = parsed_url.path.strip("/").replace("/", "_")

        if not url_path:
            url_path = "homepage"

        url_path = url_path.replace(".en", "").replace(".html", "")

        if len(url_path) > 50:
            url_path = url_path[:50]

        md_file = MARKDOWN_DIR / f"{i:03d}_{url_path}.md"

        with open(md_file, "w", encoding="utf-8") as f:
            f.write(f"# {doc.get('title', 'No Title')}\n\n")
            f.write(f"**Source URL**: {doc['url']}\n")
            f.write(f"**Crawl Depth**: {doc.get('depth_level', 0)}\n")

            if doc.get("metadata"):
                f.write("**Metadata**:\n")
                for key, val in doc["metadata"].items():
                    if val:
                        f.write(f"- {key}: {val}\n")

            f.write("\n---\n\n")
            f.write(doc.get("content", ""))

        saved_count += 1

    except Exception as e:
        failed_count += 1
        print(f"Warning: Failed to save markdown for {doc.get('url', 'unknown')}: {e}")

print(f"\nMarkdown save summary:")
print(f"  Saved  : {saved_count}")
print(f"  Failed : {failed_count}")

### 7. Quality Checks

In [ ]:
# Required metadata fields per rubric
REQUIRED_METADATA_FIELDS = [
    "property_id", "title", "address", "price",
    "bedrooms", "bathrooms", "sqft", "amenities", "agent"
]

print("Inspecting data quality...\n")

# --- Check 1: Markdown files ---
md_files = list(MARKDOWN_DIR.glob("*.md"))
print(f"1. Markdown files: {len(md_files)}")
if len(md_files) == 0:
    print("   CRITICAL: No markdown files found.")
elif len(md_files) < 5:
    print(f"   WARNING: Only {len(md_files)} files. Rubric requires 5+ entries.")
else:
    print(f"   OK ({len(md_files)} files)")

# --- Check 2: JSONL file ---
if not JSONL_PATH.exists():
    raise FileNotFoundError(f"JSONL file not found at {JSONL_PATH}")

file_size = JSONL_PATH.stat().st_size
print(f"\n2. JSONL file size: {file_size / 1024:.1f} KB")
if file_size == 0:
    print("   CRITICAL: JSONL file is empty.")
else:
    print("   OK")

# --- Load all documents ---
try:
    with open(JSONL_PATH, "r", encoding="utf-8") as f:
        all_docs = [json.loads(line) for line in f if line.strip()]
except Exception as e:
    print(f"CRITICAL: Could not parse JSONL file: {e}")
    all_docs = []

# --- Check 3: Metadata field completeness ---
print(f"\n3. Metadata field completeness (required fields: {REQUIRED_METADATA_FIELDS}):")
if not all_docs:
    print("   CRITICAL: No documents to inspect.")
else:
    field_hit_counts = {field: 0 for field in REQUIRED_METADATA_FIELDS}

    for doc in all_docs:
        metadata = doc.get("metadata", {})
        # Also check top-level keys for flat schemas
        combined = {**doc, **metadata}
        for field in REQUIRED_METADATA_FIELDS:
            if combined.get(field) not in (None, "", [], {}):
                field_hit_counts[field] += 1

    total_docs = len(all_docs)
    print(f"   Total documents: {total_docs}")
    print(f"   {'Field':<15} {'Present':>10} {'Coverage':>10}")
    print(f"   {'-'*38}")
    all_covered = True
    for field, count in field_hit_counts.items():
        coverage = count / total_docs * 100 if total_docs else 0
        status = "OK" if coverage >= 50 else "LOW"
        if coverage < 50:
            all_covered = False
        print(f"   {field:<15} {count:>10} {coverage:>9.1f}%  [{status}]")

    if all_covered:
        print("\n   All required fields present in majority of documents.")
    else:
        print("\n   WARNING: Some required metadata fields have low coverage. Check crawler extraction logic.")

# --- Check 4: Content quality samples ---
print("\n4. Content quality (random sample):")
if all_docs:
    samples = random.sample(all_docs, min(3, len(all_docs)))
    for i, doc in enumerate(samples, 1):
        content = doc.get("content", "")
        word_count = len(content.split())
        header_sample = content[:100].lower()
        is_menu_suspect = "home" in header_sample and "contact" in header_sample

        print(f"\n   Sample {i}: {doc.get('title', 'No Title')}")
        print(f"   URL    : {doc.get('url', 'N/A')}")
        print(f"   Words  : {word_count}")

        if word_count < 50:
            print("   WARNING: Very short content — likely an error or empty page.")
        elif is_menu_suspect and word_count < 150:
            print("   WARNING: Potential menu trap — content looks like navigation links only.")
        else:
            print(f"   OK — Preview: {content[:120].replace(chr(10), ' ')}...")

print("\nQuality checks complete.")

### 8. Document Structure Verification

In [ ]:
try:
    with open(JSONL_PATH, "r", encoding="utf-8") as f:
        first_line = f.readline().strip()

    if not first_line:
        print("CRITICAL: JSONL file is empty — no schema to display.")
    else:
        sample = json.loads(first_line)
        print("Document schema (first entry, truncated to 500 chars):")
        print(json.dumps(sample, indent=2, ensure_ascii=False)[:500])

        # Verify top-level keys
        print("\nTop-level keys found:", list(sample.keys()))

        missing_top = [k for k in ["url", "title", "content", "metadata", "depth_level"] if k not in sample]
        if missing_top:
            print(f"WARNING: Expected keys missing from schema: {missing_top}")
        else:
            print("All expected top-level keys present.")

except json.JSONDecodeError as e:
    print(f"ERROR: Could not parse first JSONL entry: {e}")
except Exception as e:
    print(f"ERROR: Unexpected failure reading JSONL: {e}")